# ⚛️ Módulo 3: Mecánica Molecular y Campos de Fuerza
## Actividad 3.4: Superficies de Energía Potencial

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_03_mecanica_molecular/04_superficies_energia.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comprender el concepto de Superficie de Energía Potencial (PES)
- Calcular y visualizar PES en 1D y 2D con mecánica molecular
- Identificar puntos estacionarios: mínimos y puntos de silla
- Trazar caminos de reacción mínimos (MEP)
- Relacionar la PES con la reactividad química
- Aplicar escaneos de diedros para análisis conformacional básico

---

## 📚 Introducción

La **Superficie de Energía Potencial (PES, del inglés *Potential Energy Surface*)** describe cómo varía la energía potencial de un sistema molecular en función de sus coordenadas nucleares.

### Concepto Fundamental

Para una molécula con $N$ átomos, la PES es una función de $3N - 6$ coordenadas internas (o $3N - 5$ para moléculas lineales):

$$E = E(q_1, q_2, \ldots, q_{3N-6})$$

### Tipos de Puntos Estacionarios

En un punto estacionario, el gradiente de energía es cero: $\nabla E = 0$

| Tipo | Condición | Eigenvalores Hessianos | Significado |
|------|-----------|------------------------|-------------|
| **Mínimo local** | $\nabla E = 0$ | Todos positivos | Conformero estable |
| **Mínimo global** | $\nabla E = 0$, $E$ más baja | Todos positivos | Conformero más estable |
| **Punto de silla (TS)** | $\nabla E = 0$ | Un negativo | Estado de transición |
| **Máximo** | $\nabla E = 0$ | Todos negativos | Raro, inestable |

### Scans de Coordenadas

Un **scan rígido** (rigid scan) mueve una coordenada manteniendo las demás fijas.  
Un **scan relajado** (relaxed scan) optimiza todas las demás coordenadas en cada punto.  
El scan relajado es más costoso pero más realista y se usa para caminos de reacción.

### Aplicaciones
- Análisis conformacional de moléculas flexibles
- Rutas de reacción y estados de transición
- Parametrización de campos de fuerza
- Predicción de barreras de rotación

In [ ]:
# Instalación de dependencias
!pip install rdkit-pypi numpy scipy matplotlib 2>/dev/null || \
  pip install rdkit numpy scipy matplotlib
print('✓ Dependencias instaladas')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdMolDescriptors
    from rdkit.Chem import rdForceFieldHelpers
    RDKIT_OK = True
    print('✓ RDKit disponible')
except ImportError:
    RDKIT_OK = False
    print('⚠️  RDKit no disponible')

print('✓ Importaciones completadas')

## 1. PES en 1D: Potencial de Lennard-Jones

El potencial de **Lennard-Jones** (LJ 12-6) es el ejemplo más clásico de PES en 1D:

$$V(r) = 4\varepsilon \left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

Donde:
- $\varepsilon$ = profundidad del pozo (energía de la interacción)
- $\sigma$ = distancia donde $V = 0$
- $r_{min} = 2^{1/6}\sigma$ = distancia de mínimo energía

In [ ]:
def potencial_lj(r, epsilon=1.0, sigma=1.0):
    """Potencial de Lennard-Jones 12-6."""
    return 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)

def potencial_morse(r, De=1.0, a=1.5, re=1.0):
    """Potencial de Morse para enlace covalente."""
    return De * (1 - np.exp(-a * (r - re)))**2 - De

def potencial_armonico(r, k=1.0, r0=1.0):
    """Potencial armónico (aproximación de Born-Oppenheimer)."""
    return 0.5 * k * (r - r0)**2

# Calcular y graficar
r = np.linspace(0.8, 4.0, 500)
r_lj = np.linspace(0.9, 4.0, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lennard-Jones
ax = axes[0]
for eps, label, color in [(1.0, 'ε=1.0', '#2196F3'), 
                           (0.5, 'ε=0.5', '#FF9800'),
                           (2.0, 'ε=2.0', '#4CAF50')]:
    V = potencial_lj(r_lj, epsilon=eps, sigma=1.0)
    ax.plot(r_lj, np.clip(V, -3, 5), label=label, linewidth=2.0, color=color)

r_min = 2**(1/6)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(r_min, color='red', linestyle=':', alpha=0.7, label=f'r_min = {r_min:.3f}σ')
ax.set_xlim(0.9, 4.0)
ax.set_ylim(-2.5, 3)
ax.set_xlabel('r / σ', fontsize=12)
ax.set_ylabel('V(r) / ε', fontsize=12)
ax.set_title('Potencial de Lennard-Jones\n(interacciones no covalentes)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.fill_between(r_lj, np.clip(potencial_lj(r_lj), -3, 5), -3, 
                where=r_lj > r_min, alpha=0.05, color='blue')

# Morse vs Armónico
ax = axes[1]
r_m = np.linspace(0.3, 3.5, 500)
ax.plot(r_m, potencial_morse(r_m, De=4.5, a=2.0, re=0.74), 
        label='Morse (real)', color='#2196F3', linewidth=2.5)
ax.plot(r_m, np.clip(potencial_armonico(r_m, k=18.0, r0=0.74), -0.5, 6.0), 
        label='Armónico (aprox.)', color='#FF5722', linewidth=2.0, linestyle='--')
ax.axhline(-4.5, color='gray', linestyle=':', alpha=0.6, label='Límite disociación')
ax.axvline(0.74, color='green', linestyle=':', alpha=0.7, label=f'r_eq = 0.74 Å (H₂)')
ax.set_xlim(0.3, 3.5)
ax.set_ylim(-5, 6)
ax.set_xlabel('r (Å)', fontsize=12)
ax.set_ylabel('Energía (eV)', fontsize=12)
ax.set_title('Potencial de Morse vs Armónico\n(enlace covalente H₂)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Superficies de Energía Potencial en 1D', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nPropiedades del potencial LJ (σ=1, ε=1):')
print(f'  Mínimo en r = 2^(1/6)·σ = {2**(1/6):.4f} σ')
print(f'  Profundidad = -ε = -1.0')
print(f'  V → 0 cuando r → ∞')

## 2. PES en 1D: Scan de Ángulo Diedro (Butano)

La rotación del ángulo diedro C−C−C−C del butano es el ejemplo canónico de PES torsional. Produce una serie de **conformeros** separados por barreras energéticas:

| Conformero | Diedro | Energía relativa |
|------------|--------|------------------|
| Anti | 180° | 0.0 kcal/mol (más estable) |
| Gauche+ | 60° | ~0.9 kcal/mol |
| Eclipsado parcial | 120° | ~3.6 kcal/mol (silla) |
| Eclipsado total | 0°/360° | ~6.0 kcal/mol (máximo) |

In [ ]:
def pes_butano_mm(phi_deg):
    """
    Aproximación analítica de la PES del butano (diedro C-C-C-C).
    Basado en parámetros AMBER/OPLS típicos.
    Energía en kcal/mol, ángulo en grados.
    """
    phi = np.radians(phi_deg)
    # Potencial de Fourier: V = sum_n [V_n/2 * (1 + cos(n*phi - gamma_n))]
    V1 = 1.40 * (1 + np.cos(phi))
    V2 = 0.54 * (1 - np.cos(2*phi))
    V3 = 0.20 * (1 + np.cos(3*phi))
    return V1 + V2 + V3

phi = np.linspace(-180, 180, 360)
E = pes_butano_mm(phi)
E -= E.min()  # Referencia al mínimo

# Encontrar mínimos y máximos
from scipy.signal import argrelextrema
minimos_idx = argrelextrema(E, np.less, order=15)[0]
maximos_idx = argrelextrema(E, np.greater, order=15)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PES del butano
ax = axes[0]
ax.plot(phi, E, 'b-', linewidth=2.5)
ax.scatter(phi[minimos_idx], E[minimos_idx], color='green', s=120, zorder=5,
          label='Mínimos (conformeros)', marker='v')
ax.scatter(phi[maximos_idx], E[maximos_idx], color='red', s=120, zorder=5,
          label='Máximos (eclipsados)', marker='^')

# Etiquetas
for idx in minimos_idx:
    label = 'Anti' if abs(phi[idx]) > 100 else 'Gauche'
    ax.annotate(f'{label}\n{phi[idx]:.0f}°\n{E[idx]:.2f} kcal/mol',
               (phi[idx], E[idx]), textcoords='offset points', xytext=(0, -40),
               ha='center', fontsize=8, color='green',
               arrowprops=dict(arrowstyle='->', color='green', lw=1))

ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Ángulo diedro φ (°)', fontsize=12)
ax.set_ylabel('Energía relativa (kcal/mol)', fontsize=12)
ax.set_title('PES Torsional del Butano\n(diedro C−C−C−C)', fontsize=12, fontweight='bold')
ax.set_xticks([-180, -120, -60, 0, 60, 120, 180])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Representación de conformeros
ax = axes[1]
conformeros = {
    'Anti (180°)': (180, E[np.argmin(np.abs(phi-180))]),
    'Gauche+ (60°)': (60, E[np.argmin(np.abs(phi-60))]),
    'Gauche- (-60°)': (-60, E[np.argmin(np.abs(phi+60))]),
    'Ecl. parcial (120°)': (120, E[np.argmin(np.abs(phi-120))]),
    'Ecl. total (0°)': (0, E[np.argmin(np.abs(phi-0))]),
}

nombres = list(conformeros.keys())
energias = [v[1] for v in conformeros.values()]
colores_c = ['#4CAF50' if e < 1.5 else '#FF9800' if e < 4 else '#F44336' for e in energias]

ax.bar(nombres, energias, color=colores_c, alpha=0.85, edgecolor='white')
ax.set_ylabel('Energía relativa (kcal/mol)', fontsize=12)
ax.set_title('Energías de Conformeros\ndel Butano', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(True, alpha=0.3, axis='y')
for i, (nom, e) in enumerate(zip(nombres, energias)):
    ax.text(i, e + 0.1, f'{e:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Análisis PES Torsional: Butano', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 RESUMEN DE CONFORMEROS DEL BUTANO:')
for nom, (ang, e) in conformeros.items():
    tipo = 'Mínimo' if e < 1.5 else 'Punto de silla' if e < 4.5 else 'Máximo'
    print(f'  {nom}: {e:.2f} kcal/mol ({tipo})')

## 3. PES en 2D con RDKit

Para moléculas con más de un diedro flexible, necesitamos una PES 2D. Ejemplo: **propanol** con dos diedros (rotación del OH y del esqueleto C−C−C−O).

In [ ]:
def scan_diedro_rdkit(smiles, diedro_idx, n_puntos=36, verbose=False):
    """
    Realiza un scan rígido de un diedro usando RDKit MMFF94.
    Retorna ángulos y energías.
    """
    if not RDKIT_OK:
        return None, None

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)

    ff = AllChem.MMFFGetMoleculeForceField(
        mol, AllChem.MMFFGetMoleculeProperties(mol))

    angulos = np.linspace(-180, 175, n_puntos)
    energias = []

    for ang in angulos:
        # Restringir el diedro al ángulo deseado
        ff.MMFFAddTorsionConstraint(*diedro_idx, False, ang, ang, 1e5)
        ff.Minimize(maxIts=200)
        energias.append(ff.CalcEnergy())
        # Limpiar constraint (reiniciar ff)
        ff = AllChem.MMFFGetMoleculeForceField(
            mol, AllChem.MMFFGetMoleculeProperties(mol))

    energias = np.array(energias)
    energias -= energias.min()
    return angulos, energias

def pes_2d_analitica(phi1, phi2):
    """
    PES 2D analítica para demostración (tipo n-butanol).
    phi1: diedro O-C-C-C, phi2: diedro C-C-C-C
    """
    # Componente torsional C-C-C-C (butano)
    p2 = np.radians(phi2)
    V_cc = 1.40*(1+np.cos(p2)) + 0.54*(1-np.cos(2*p2)) + 0.20*(1+np.cos(3*p2))

    # Componente torsional O-C-C-C (ligeramente diferente)
    p1 = np.radians(phi1)
    V_oc = 0.80*(1+np.cos(p1)) + 0.40*(1-np.cos(2*p1)) + 0.30*(1+np.cos(3*p1))

    # Acoplamiento torsional ligero
    V_coupling = 0.15 * np.cos(p1) * np.cos(p2)

    V = V_cc + V_oc + V_coupling
    return V - V.min()

# PES 2D
phi1_arr = np.linspace(-180, 180, 100)
phi2_arr = np.linspace(-180, 180, 100)
PHI1, PHI2 = np.meshgrid(phi1_arr, phi2_arr)
Z = pes_2d_analitica(PHI1, PHI2)

fig = plt.figure(figsize=(16, 6))

# Mapa de contorno
ax1 = fig.add_subplot(131)
cp = ax1.contourf(PHI1, PHI2, Z, levels=25, cmap='RdYlGn_r')
ax1.contour(PHI1, PHI2, Z, levels=25, colors='white', alpha=0.3, linewidths=0.5)
plt.colorbar(cp, ax=ax1, label='Energía (kcal/mol)')
ax1.set_xlabel('φ₁ O−C−C−C (°)', fontsize=11)
ax1.set_ylabel('φ₂ C−C−C−C (°)', fontsize=11)
ax1.set_title('PES 2D\n(mapa de contornos)', fontsize=11, fontweight='bold')

# Superficie 3D
ax2 = fig.add_subplot(132, projection='3d')
surf = ax2.plot_surface(PHI1, PHI2, np.clip(Z, 0, 8), cmap='RdYlGn_r',
                        alpha=0.85, linewidth=0)
ax2.set_xlabel('φ₁ (°)', fontsize=9)
ax2.set_ylabel('φ₂ (°)', fontsize=9)
ax2.set_zlabel('E (kcal/mol)', fontsize=9)
ax2.set_title('PES 2D (3D)', fontsize=11, fontweight='bold')
ax2.view_init(elev=25, azim=45)

# Corte 1D a phi2=180 (diedro C-C-C-C = anti)
ax3 = fig.add_subplot(133)
idx_anti = np.argmin(np.abs(phi2_arr - 180))
ax3.plot(phi1_arr, Z[idx_anti, :], 'b-', linewidth=2, label='φ₂=180° (Anti)')
idx_gauche = np.argmin(np.abs(phi2_arr - 60))
ax3.plot(phi1_arr, Z[idx_gauche, :], 'r--', linewidth=2, label='φ₂=60° (Gauche)')
ax3.set_xlabel('φ₁ O−C−C−C (°)', fontsize=11)
ax3.set_ylabel('Energía relativa (kcal/mol)', fontsize=11)
ax3.set_title('Cortes 1D de la PES 2D', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.suptitle('Superficie de Energía Potencial 2D — n-Butanol', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Puntos Estacionarios y Caminos de Reacción

El **Camino de Mínima Energía (MEP, *Minimum Energy Path*)** conecta dos mínimos pasando por el punto de silla de menor energía.

In [ ]:
from scipy.signal import argrelextrema

def analizar_pes_1d(phi, E, nombre='Sistema'):
    """
    Analiza una PES 1D: encuentra y clasifica todos los puntos estacionarios.
    """
    min_idx = argrelextrema(E, np.less, order=10)[0]
    max_idx = argrelextrema(E, np.greater, order=10)[0]

    print(f'\n📊 ANÁLISIS DE PUNTOS ESTACIONARIOS — {nombre}')
    print('='*55)

    print('\n🟢 MÍNIMOS (conformeros estables):')
    for i, idx in enumerate(min_idx):
        print(f'  Mínimo {i+1}: φ = {phi[idx]:7.1f}°, E = {E[idx]:.3f} kcal/mol')

    print('\n🔴 MÁXIMOS (puntos de silla / barreras):')
    for i, idx in enumerate(max_idx):
        print(f'  Máximo {i+1}: φ = {phi[idx]:7.1f}°, E = {E[idx]:.3f} kcal/mol')

    # Calcular barreras de activación
    if len(min_idx) > 1 and len(max_idx) > 0:
        print('\n⚡ BARRERAS DE ACTIVACIÓN:')
        E_min_global = E[min_idx].min()
        for idx in max_idx:
            barrera = E[idx] - E_min_global
            print(f'  Barrera en φ={phi[idx]:.1f}°: ΔE‡ = {barrera:.3f} kcal/mol')

    return min_idx, max_idx

# Analizar butano
phi_b = np.linspace(-180, 180, 360)
E_b = pes_butano_mm(phi_b)
E_b -= E_b.min()

min_idx, max_idx = analizar_pes_1d(phi_b, E_b, 'Butano (diedro C-C-C-C)')

# Visualizar con anotaciones de estados de transición
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(phi_b, E_b, 'b-', linewidth=2.5, label='PES butano')
ax.fill_between(phi_b, 0, E_b, alpha=0.08, color='blue')

ax.scatter(phi_b[min_idx], E_b[min_idx], color='#4CAF50', s=150, zorder=5,
          marker='v', label='Mínimos (IS)', linewidth=1.5, edgecolors='white')
ax.scatter(phi_b[max_idx], E_b[max_idx], color='#F44336', s=150, zorder=5,
          marker='^', label='Máximos (TS)', linewidth=1.5, edgecolors='white')

# Flecha de barrera de activación
idx_ts = max_idx[np.argmin(np.abs(phi_b[max_idx] - 120))]
idx_min = min_idx[np.argmin(np.abs(phi_b[min_idx] - 60))]
ax.annotate('', xy=(phi_b[idx_ts], E_b[idx_ts]),
           xytext=(phi_b[idx_ts], E_b[idx_min]),
           arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
ax.text(phi_b[idx_ts]+8, (E_b[idx_ts]+E_b[idx_min])/2,
       f'ΔE‡={E_b[idx_ts]-E_b[idx_min]:.2f}\nkcal/mol',
       fontsize=9, color='purple', fontweight='bold')

ax.set_xlabel('Ángulo diedro φ C−C−C−C (°)', fontsize=12)
ax.set_ylabel('Energía relativa (kcal/mol)', fontsize=12)
ax.set_title('PES del Butano: Puntos Estacionarios y Barreras de Activación',
            fontsize=12, fontweight='bold')
ax.set_xticks([-180, -120, -60, 0, 60, 120, 180])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Scan con RDKit: Molécula Real

Aplicamos el scan torsional al **etano** y al **propano** usando el campo de fuerza MMFF94.

In [ ]:
def scan_torsion_rdkit(smiles, nombre, diedro_atoms, n_puntos=36):
    """
    Scan torsional usando RDKit MMFF94.
    diedro_atoms: tupla de 4 índices de átomos.
    """
    if not RDKIT_OK:
        print('RDKit no disponible. Usando datos simulados.')
        phi = np.linspace(-180, 175, n_puntos)
        # Simular con potencial de Fourier + ruido pequeño
        phi_r = np.radians(phi)
        E = (1.4*(1+np.cos(phi_r)) + 0.54*(1-np.cos(2*phi_r)) + 
             0.20*(1+np.cos(3*phi_r)))
        E -= E.min()
        E += np.random.normal(0, 0.02, len(E))
        return phi, E

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)

    props = AllChem.MMFFGetMoleculeProperties(mol)
    angulos = np.linspace(-180, 175, n_puntos)
    energias = []

    mol_work = Chem.RWMol(mol)
    conf = mol_work.GetConformer()

    for ang in angulos:
        mol_tmp = Chem.RWMol(mol)
        ff = AllChem.MMFFGetMoleculeForceField(mol_tmp, props)
        if ff:
            ff.MMFFAddTorsionConstraint(*diedro_atoms, False, ang, ang, 5e4)
            ff.Minimize(maxIts=200)
            energias.append(ff.CalcEnergy())
        else:
            energias.append(np.nan)

    E = np.array(energias)
    E -= np.nanmin(E)
    return angulos, E

# Scan del etano (H-C-C-H)
np.random.seed(42)
phi_et, E_et = scan_torsion_rdkit('CC', 'Etano', (0,1,2,3))

# Scan del propano (C-C-C-H)
phi_pr, E_pr = scan_torsion_rdkit('CCC', 'Propano', (0,1,2,3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (phi_s, E_s, nombre_s, color) in zip(axes, [
    (phi_et, E_et, 'Etano (H−C−C−H)', '#2196F3'),
    (phi_pr, E_pr, 'Propano (C−C−C−H)', '#4CAF50'),
]):
    ax.plot(phi_s, E_s, 'o-', color=color, linewidth=2, markersize=5)
    ax.fill_between(phi_s, 0, E_s, alpha=0.1, color=color)
    ax.set_xlabel('Ángulo diedro (°)', fontsize=12)
    ax.set_ylabel('Energía relativa (kcal/mol)', fontsize=12)
    ax.set_title(f'Scan Torsional — {nombre_s}', fontsize=12, fontweight='bold')
    ax.set_xticks([-180, -120, -60, 0, 60, 120, 180])
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
    E_max = E_s.max()
    ax.text(0.02, 0.95, f'Barrera máx: {E_max:.2f} kcal/mol',
           transform=ax.transAxes, fontsize=10, va='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle('Scans Torsionales con MMFF94 (RDKit)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Calcula y grafica la PES 1D del **metanol** (diedro H−O−C−H) usando la función `pes_butano_mm` como base, pero ajustando los coeficientes $V_1 = 0.45$, $V_2 = 0.10$, $V_3 = 0.35$ kcal/mol. ¿Cuántos mínimos tiene? ¿Cuál es la barrera de rotación?

### Ejercicio 2 (Intermedio)
Construye una PES 2D para el **1,2-dicloroetano** variando los ángulos diedro Cl−C−C−Cl ($\phi_1$) y H−C−C−Cl ($\phi_2$) de -180° a 180°. ¿Cuántos mínimos observas? ¿Cuál corresponde al conformero anti y cuál al gauche?

### Ejercicio 3 (Avanzado)
Usa `scan_torsion_rdkit` para calcular la PES del **2-buteno** (diedro C=C−C−C). Compara la PES del cis-2-buteno vs trans-2-buteno. ¿Cuál tiene mayor barrera rotacional? Justifica en términos de interacciones estéricas.

In [ ]:
# Ejercicio 1: PES del metanol
phi_met = np.linspace(-180, 180, 360)
V1, V2, V3 = 0.45, 0.10, 0.35
p = np.radians(phi_met)
E_met = V1*(1+np.cos(p)) + V2*(1-np.cos(2*p)) + V3*(1+np.cos(3*p))
E_met -= E_met.min()

plt.figure(figsize=(8, 4))
plt.plot(phi_met, E_met, 'r-', linewidth=2)
plt.xlabel('Ángulo diedro H−O−C−H (°)')
plt.ylabel('Energía relativa (kcal/mol)')
plt.title('PES del Metanol')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Barrera máxima: {E_met.max():.3f} kcal/mol')
# Tu código para ejercicios 2 y 3 aquí...

## 7. Referencias

1. Leach, A. R. (2001). *Molecular Modelling: Principles and Applications*, 2nd ed. Pearson.
2. Jensen, F. (2017). *Introduction to Computational Chemistry*, 3rd ed. Wiley.
3. Cramer, C. J. (2004). *Essentials of Computational Chemistry*. Wiley.
4. RDKit Documentation: https://www.rdkit.org/docs/
5. Schlegel, H. B. (2011). Exploring potential energy surfaces for chemical reactions. *J. Comput. Chem.*, 32(12), 2429–2437.

---

## 📚 Recursos Adicionales

### Herramientas
- [RDKit Cookbook](https://www.rdkit.org/docs/Cookbook.html) — Recetas de código para MM
- [ASE (Atomic Simulation Environment)](https://wiki.fysik.dtu.dk/ase/) — Scans y NEB
- [MOLPRO NEB Tutorial](https://www.molpro.net/) — Caminos de reacción QM

### Visualización interactiva
- Avogadro2 permite visualizar scans torsionales de forma interactiva
- Molden puede mostrar la variación geométrica a lo largo de una PES

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Definir PES y explicar su importancia en química computacional
- ✅ Identificar mínimos, máximos y puntos de silla en una PES
- ✅ Calcular y graficar PES en 1D (scan de diedro) con numpy/scipy
- ✅ Construir una PES 2D y visualizarla como mapa de contornos
- ✅ Calcular barreras de activación entre conformeros
- ✅ Usar RDKit para realizar scans torsionales reales

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 3.4: Superficies de Energía Potencial**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_3.3-Optimización_Geometrías-blue.svg)](03_optimizacion_geometrias.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_3.5_➡️-Análisis_Conformacional-green.svg)](05_analisis_conformacional.ipynb)

---

📚 **[Volver al Módulo 3](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G - 2026*

</div>